# Lighthouse - Text Classifier Practice

## Part A: watch (about 30 min)

Watch this video: **Text Classification with Python: Build and Compare Three Text Classifiers**
https://www.youtube.com/watch?v=eOu-h_XxjHQ

Do not code along with the video, just follow along with his code, understand what he is doing, and try to remember the methods and libraries he uses. If you type what he types without understanding, it will be hard to understand why it works.

He classifies text messages as spam or not spam. Pay attention to:
- what TF-IDF is doing and why (around the beginning)
- why he splits the data into train and test
- what `random_state=42` is for
- his data is lopsided, 4,825 of one class and 747 of the other, and he explains
  why that matters. The data we will be using is will likely be lopsided the same way.
- near the end he changes some settings and the score gets worse. Please note that.

## Part B: recreating

We will be doing the same thing he did, but on our dark patterns data, in this notebook.

The data loading is done for you below, because parsing that CSV is annoying and
it isn't the point. Everything after that is yours.

You may get stuck. Rewatch the relevant bits of the video as needed,
or Google the error message. It's also likely that you may forget the library or method, and that is easily googleable. If you're stuck more than 20 minutes, please message me in discord before consulting AI. Even if I use AI to answer your question, I am able to filter that information and ensure you're learning with what I tell you, rather than AI giving you all the info. Please try not to use AI for this.

---
# Setup — already done for you

Run these two cells and take a quick glance on their comments and how the code works.

In [2]:
# Load the dataset from the Mathur et al. 2019 paper
# (1,818 real dark patterns collected from ~11,000 shopping websites)
import pandas as pd

URL = ("https://raw.githubusercontent.com/aruneshmathur/dark-patterns/"
       "master/data/final-dark-patterns/dark-patterns.csv")

df = pd.read_csv(URL)
df = df[["Pattern String", "Pattern Category", "Pattern Type"]]
df.columns = ["text", "category", "subtype"]

# Clean: drop rows with no text, drop exact duplicates
df = df.dropna(subset=["text"]).copy()
df["text"] = df["text"].str.strip()
df = df[df["text"] != ""]
df = df.drop_duplicates(subset="text")

# Collapse the paper's 7 categories into 5 buckets for this exercise.
# (Our real taxonomy has six categories plus a "none" class. More on that later.)
def make_label(row):
    if row["subtype"] == "Confirmshaming":
        return "confirmshaming"
    if row["category"] == "Urgency":
        return "urgency"
    if row["category"] == "Scarcity":
        return "scarcity"
    if row["category"] == "Social Proof":
        return "social_proof"
    return "other"

df["label"] = df.apply(make_label, axis=1)

print("Ready. Rows:", len(df))
print("(The paper has 1,818 rows. Dropping exact duplicate strings leaves fewer.)")

Ready. Rows: 1178
(The paper has 1,818 rows. Dropping exact duplicate strings leaves fewer.)


In [3]:
# Look at what you're working with
print("--- 10 real dark patterns ---")
for t in df["text"].sample(10, random_state=1):
    print(" *", t)

print()
print("--- How many of each category ---")
print(df["label"].value_counts())

--- 10 real dark patterns ---
 * Limited Time Offer!
 * 35% OFF EVERYTHING* EXCL SALE & BEAUTY ENTER: DOLL35 - HURRY! ENDS IN  16H 59M 06S
 * Save $148.98AUD – 49% Off
 * No thanks, I don't want to build one.. I'd rather kick rocks!
 * Availability: 5 items left
 * 24 people currently interested
 * Limited Time Offers
 * "Your purchase entitles you to the following special"
 * Only 1 Item Left!
 * Offer ends in 03 days 19 hours : 54 mins : 22 secs

--- How many of each category ---
label
scarcity          418
social_proof      312
urgency           210
confirmshaming    129
other             109
Name: count, dtype: int64


---
# YOUR WORK STARTS HERE

You have a dataframe called `df` with two columns that matter:

| column | what it is |
|---|---|
| `df["text"]` | the UI text, e.g. `"Only 2 left in stock!"` |
| `df["label"]` | the answer, one of 5 categories |

This is the same shape as the video: a text column and a label column.
His were `message` and `category`. Yours are `text` and `label`.

---

## Task 1 — Split into training and testing

Same as the video. Use 20% for testing and set a random state so it's repeatable. If you cannot remember the method or command used, try to find it in his video.

In [4]:
# YOUR CODE
# Splitting the two components text and label
X = df['text']
y = df['label']
len(X)


1178

In [5]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 41) #splitting the data 20 precent will be used to test and 80 will be used to train.
#random_state allows for this randomization to stay static amongst all models for fair analyis.

In [6]:
len(X_train) #the rest is for our training purposes

942

## Task 2 — Build a pipeline and train it

Same structure as the video: a TF-IDF vectorizer, then a classifier.

Use **`LogisticRegression`** as your classifier (from `sklearn.linear_model`).
It's not one of the three he used, but it works the same way and it's
what our project uses as its baseline.

Two settings you'll want on it: `max_iter=1000` so it doesn't complain about
not finishing, and `class_weight="balanced"` because our data is lopsided
(remember his spam/ham problem).

In [7]:
# YOUR CODE
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report


In [8]:
pipeMNB = Pipeline([ ('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())  ])
pipeCNB = Pipeline([ ('tfidf', TfidfVectorizer()), ('clf', ComplementNB())  ])
pipeSVC = Pipeline([ ('tfidf', TfidfVectorizer()), ('clf', LinearSVC())  ])

In [9]:
pipeMNB.fit(X_train, y_train)
predictMNB = pipeMNB.predict(X_test)

## Task 3 — How good is it?

Predict on your test set and print a `classification_report`.

Look at the **f1-score** (you can google what this is) column. Which categories does it do well on? Which is worst?

You would expect the categories with the least data to do worst. Check whether that
actually holds here. One of the small categories does suspiciously well — before you
accept that number, scroll up and read a few of that category's actual sentences.
Ask yourself whether the model learned the concept or just memorized a template.

In [10]:
# YOUR CODE
print(f"MNB: {accuracy_score(y_test, predictMNB): .2f}") #.92
print(f"CNB: {accuracy_score(y_test, predictCNB): .2f}") #.95
print(f"SVC: {accuracy_score(y_test, predictSVC): .2f}") #.96


MNB:  0.92


NameError: name 'predictCNB' is not defined

In [ ]:
print(classification_report(y_test, predictSVC))

## Task 4 — Try to break it

The video ends by writing a custom message and predicting on it. Do that here.

First try sentences written the way the training data is written:
- `"Only 2 left in stock!"`
- `"Hurry, offer ends at midnight"`

Then, and this is the real task, **write dark patterns in your own words.**
Not copied from a shopping site. For example, instead of "Only 2 left in stock",
try "better grab it, barely any left".

**Find at least one the model gets wrong.** They're in there.

In [ ]:
# YOUR CODE
msg = "Hurry, offer ends at midnight"
pipeSVC.predict([msg]) #this outputs the correct result for all 3 models



In [ ]:
msg2 = "Are you sure you want to stop protecting your family?"
pipeSVC.predict([msg2]) #this output is incorrect

---
# 📬 Take note of the below information when you're done, it'll serve as your deliverable

**1.** What f1-score did you get overall? Which category was worst?

**2.** Paste one dark-pattern sentence **your model got wrong**, and what it
guessed instead.

**3.** Our taxonomy has six categories plus `none`. This data only gave you five
buckets, and two of our six are missing entirely. Take a guess at which two, and why
a dataset like this one could never contain them.

**4.** One thing that confused you from either this notebook or the video

---

### Stuck?

- **Error you don't understand?** Paste the line of it into Google and see if the answer is there.
- **Still stuck after 20 minutes?** DM me on discord.
- The video serves as a hint bank.
- Please please please do not have AI do this for you. If you do ask AI a question, carefully prompt it to only answer the small thing you asked in a vacuum.